# 1. Templates
For chat / instruction models, their corresponding tokenizers typically include the chat template used to train the model.

## 1.1 Build-in Template

In [1]:
from transformers import AutoTokenizer

In [2]:
model_name = "microsoft/phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

***
Take a quick look at chat template of `phi3` pretrained model

In [3]:
from rich import print as pprint

In [4]:
pprint(tokenizer.chat_template)

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}

The templates are written in Jinja which generates strings using python variables.<br>
Let's try it out to change an conversation with template.

In [5]:
messages = [
    {'role': 'system', 'content': 'You are a helpful AI assistant.'},
    {'role': 'user',  'content': 'What is the capital of Argentina?'},
    {'role': 'assistant', 'content': 'Buenos Aires.'}
]
### messages are in conversation format

In [6]:
formatted = tokenizer.apply_chat_template(
    conversation=messages,
    tokenize=False,
    add_generation_prompt=False # our messages already contains both user's prompt and assistant's completion
)

pprint(formatted)

<|system|>
You are a helpful AI assistant.<|end|>
<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>
Buenos Aires.<|end|>
<|endoftext|>

The `tokenizer.apply_chat_template()` method only accept conversational format messages but there exists <u>instructional format messages</u>. To deal with such situation, we need to convert the instructional messages to conversational messages then apply to chat template method.

In [7]:
messages = [
    {'prompt':'What is the capital of Argentina?', 'completion':'Buenos Aires.'},
    {'prompt':'What is the capital of the United States?', 'completion':'Washington D.C.'},
]

### messages are in instructional format

In [8]:
### custom function to transform instructional format to conversational format

def instruction_to_conversation(msgs):
    outputs = []
    for msg in msgs:
        outputs.append({'role':'user', 'content':msg['prompt']})
        outputs.append({"role": "assistant", "content":msg["completion"]})
    return outputs

In [9]:
converted_messages = instruction_to_conversation(messages)
pprint(converted_messages)

[
    {'role': 'user', 'content': 'What is the capital of Argentina?'},
    {'role': 'assistant', 'content': 'Buenos Aires.'},
    {'role': 'user', 'content': 'What is the capital of the United States?'},
    {'role': 'assistant', 'content': 'Washington D.C.'}
]

In [10]:
formatted = tokenizer.apply_chat_template(
    conversation=converted_messages,
    tokenize=False,
    add_generation_prompt=False # our messages already contains both user's prompt and assistant's completion
)

pprint(formatted)

<|user|>
What is the capital of Argentina?<|end|>
<|assistant|>
Buenos Aires.<|end|>
<|user|>
What is the capital of the United States?<|end|>
<|assistant|>
Washington D.C.<|end|>
<|endoftext|>

****
Before `trl` package version 0.27 there are build-in functions to implement template for training chat / instruction model, usage like below.
```python
from trl.extras.dataset_formatting import instructions_formatting_function, conversations_formatting_function
```
Author of the book has copied the original implementation code in separated custom `trl_extras_dataset_formatting` package.
****

In [11]:
from trl_extras_dataset_formatting import instructions_formatting_function, conversations_formatting_function

## 1.2 Work with Dataset
This section provide examples of applying chat_template to the `Datasets` package. Previous `instruction_to_conversation` works on list of prompts and completions. However the `Dataset` of `Transformers` is in format of 
```python
{
    'prompt':<List of prompts>,
    'translation':<List of translations>,
    'completion':<List of completion>
}
```
Therefore, we need a function that could deal with `Dataset`.

In [12]:
# Adapted from trl.extras.dataset_formatting.instructions_formatting_function
# Converts dataset from prompt/completion format (not supported anymore)
# to the conversational format
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {'messages': output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}

# 2. Tokenizer
Each base model has corresponding tokenizer. The **tokenizer and its base model share an unbreakable bond**. You must use them in tandem!<br>
Let's try tokenizer!

In [13]:
tokenizer("Let's tokenize this sentence!")

{'input_ids': [2803, 29915, 29879, 5993, 675, 445, 10541, 29991], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}

***
There are 4 vocabularies in the sentence but the output from tokenizer has 8 token-IDs because tokenizer transforming from subword instead of normal vocabularies. <br>
There is `attention_mask` field from the output of tokenizer. The `attention_mask` is used to indicate which tokens should be kept (indicate as 1) or ignored (indicate as 0). Most obivously, <u>padding tokens</u> will be ignored and will have zero values.

## 2.1 Token and Vocabulary
The tokenizer's **vocabulary** is the list of every possible token a model can handle. Each token corresponds to a unique **Token ID** which serves as an **index to lookup table of embeddings**.<br>
Nowadays models have embedding layers longer than what is actually required by the correspnding vocabulry. We can assign our new token to "empty slot" of the embedding layers.

In [14]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

In [15]:
len(tokenizer), config.vocab_size

(32011, 32064)

****
Yes! There are embedding layer is longer than vocabulary size.

In [16]:
sorted(tokenizer.vocab.items(), key=lambda t: -t[1])[:20]

[('<|user|>', 32010),
 ('<|placeholder6|>', 32009),
 ('<|placeholder5|>', 32008),
 ('<|end|>', 32007),
 ('<|system|>', 32006),
 ('<|placeholder4|>', 32005),
 ('<|placeholder3|>', 32004),
 ('<|placeholder2|>', 32003),
 ('<|placeholder1|>', 32002),
 ('<|assistant|>', 32001),
 ('<|endoftext|>', 32000),
 ('给', 31999),
 ('弘', 31998),
 ('收', 31997),
 ('왕', 31996),
 ('黃', 31995),
 ('还', 31994),
 ('边', 31993),
 ('べ', 31992),
 ('げ', 31991)]

****
You may find special tokens in form of `<|special token|>`. There are also special placeholder token for extension. We could get all special tokens by calling `all_special_tokens` or `special_tokens_map()`

In [17]:
tokenizer.all_special_tokens

['<s>', '<|endoftext|>', '<unk>']

In [18]:
tokenizer.special_tokens_map

{'bos_token': '<s>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<unk>',
 'pad_token': '<|endoftext|>'}

In [19]:
tokenizer.bos_token, tokenizer.eos_token, tokenizer.unk_token, tokenizer.pad_token

('<s>', '<|endoftext|>', '<unk>', '<|endoftext|>')

# 3. Data Collators
The **data collators** are responsible for <u>stitching together multiple data points into a mini-batch</u>. We would not use **data collators** indepently but with `SFTTrainer` class. We could specialize `data_collator` argument of  `SFTTrainer` class.

In [20]:
from torch.utils.data import DataLoader
### We load the dataset for later usage.
from datasets import load_dataset, Dataset

dataset = load_dataset("dvgodoy/yoda_sentences", split="train")
dataset = dataset.rename_column("sentence", "prompt").rename_column("translation_extra", "completion")
# converts prompt/completion pairs to conversational messages
dataset = dataset.map(format_dataset)
dataset = dataset.remove_columns(["prompt", "completion", "translation"])

In [21]:
dataset[:2]

{'messages': [[{'content': 'The birch canoe slid on the smooth planks.',
    'role': 'user'},
   {'content': 'On the smooth planks, the birch canoe slid. Yes, hrrrm.',
    'role': 'assistant'}],
  [{'content': 'Glue the sheet to the dark blue background.', 'role': 'user'},
   {'content': 'Glue the sheet to the dark blue background, you must.',
    'role': 'assistant'}]]}

In [22]:
len(dataset)

720

In [23]:
dataset.column_names

['messages']

In [24]:
formatting_func = conversations_formatting_function(tokenizer, messages_field='messages')
dataset = dataset.map(lambda row: {'text': formatting_func(row)}, batched=True, batch_size=32)

/tmp/ipykernel_51082/347937682.py:1: FutureWarning: `conversations_formatting_function` is deprecated and will be removed in TRL 0.27. Please use `tokenizer.apply_chat_template()` directly instead.
  formatting_func = conversations_formatting_function(tokenizer, messages_field='messages')


In [25]:
dataset.column_names

['messages', 'text']

In [26]:
pprint(dataset['text'][:2])

[
    '<|user|>\nThe birch canoe slid on the smooth planks.<|end|>\n<|assistant|>\nOn the smooth planks, the birch 
canoe slid. Yes, hrrrm.<|end|>\n<|endoftext|>',
    '<|user|>\nGlue the sheet to the dark blue background.<|end|>\n<|assistant|>\nGlue the sheet to the dark blue 
background, you must.<|end|>\n<|endoftext|>'
]

In [27]:
tokenized_dataset = dataset.map(lambda row: tokenizer(row['text'])).select_columns(['input_ids'])

In [28]:
pprint(tokenized_dataset['input_ids'][:2])

[
    [
        32010,
        450,
        29773,
        305,
        508,
        7297,
        2243,
        333,
        373,
        278,
        10597,
        715,
        1331,
        29889,
        32007,
        32001,
        1551,
        278,
        10597,
        715,
        1331,
        29892,
        278,
        29773,
        305,
        508,
        7297,
        2243,
        333,
        29889,
        3869,
        29892,
        298,
        21478,
        1758,
        29889,
        32007,
        32000
    ],
    [
        32010,
        8467,
        434,
        278,
        9869,
        304,
        278,
        6501,
        7254,
        3239,
        29889,
        32007,
        32001,
        8467,
        434,
        278,
        9869,
        304,
        278,
        6501,
        7254,
        3239,
        29892,
        366,
        1818,
        29889,
        32007,
        32000
    ]
]

In [29]:
tokenizer.decode(tokenized_dataset['input_ids'][0])

'<|user|> The birch canoe slid on the smooth planks.<|end|><|assistant|> On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|><|endoftext|>'

In [30]:
tokenizer.decode(tokenized_dataset['input_ids'][1])

'<|user|> Glue the sheet to the dark blue background.<|end|><|assistant|> Glue the sheet to the dark blue background, you must.<|end|><|endoftext|>'

## 3.1 Common build-in data collators
****
Then we could dive into **data collators** now.<br>
We would only discuss two **data collators** here:
- `DataCollatorForLanguageModeling` is the **default** collator for the `SFTTrainer` class
- `DataCollatorForCompletionOnlyLM` to <u>train only on the model's answer (completion)</u>
****

### 3.1.1 DataCollatorWithPadding

In [31]:
from transformers import DataCollatorWithPadding

In [32]:
padding_collator = DataCollatorWithPadding(tokenizer)
padding_dataloader = DataLoader(tokenized_dataset, batch_size=2, collate_fn=padding_collator)

In [33]:
padding_batch = next(iter(padding_dataloader))
pprint(padding_batch)

{
    'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000],
        [32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007, 32000]]),
    'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
}

In [34]:
(len(tokenized_dataset['input_ids'][0]), len(tokenized_dataset['input_ids'][1]))

(38, 28)

You may observed that the `DataCollatorWithPadding` padded the second `input_ids` because the second one is 10 ids shorter that the first one.

### 3.1.2 DataCollatorForLanguageModeling

In [35]:
from transformers import DataCollatorForLanguageModeling

In [36]:
lm_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
lm_dataloader = DataLoader(tokenized_dataset, batch_size=2, collate_fn=lm_collator)

`mlm` argument in `DataCollatorForLanguageModeling()` is the flag indicating whether it is **masked language modeling (MLM)** task where tokens are randomly removed (masked) from input, so the model is trained to predict which token should fill in the blank. **MLM** and **next-sentence prediction (NSP)** tasks are main tasks for pretraining **encoder-based model** (BERT).

In [37]:
lm_batch = next(iter(lm_dataloader))
pprint(lm_batch)

{
    'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000],
        [32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007, 32000]]),
    'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]),
    'labels': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007,  -100],
        [ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007,  -100]])
}

You may find that there is no shifting for `labels` because `SFTTrainer` handles the shifting.

In [43]:
valid_token_indexes = (lm_batch['labels'][0] >= 0)
valid_tokens = lm_batch['labels'][0][valid_token_indexes]
tokenizer.decode(valid_tokens)

'<|user|> The birch canoe slid on the smooth planks.<|end|><|assistant|> On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>'

### 3.1.3 DataCollatorForCompletionOnlyLM
In previous sections, we learnt that we need to provide `chat template` or `instruction template` for fine tuning chat / instructional pretrained model. There is data collator `DataCollatorForCompletionOnlyLM` from `trl` package that provide masking on template. 
```python
from trl import DataCollatorForCompletionOnlyLM
```
However, `DataCollatorForCompletionOnlyLM` has been removed from `trl` package since version 0.20. The original code was copied in the custom `trl_data_collator` package.<br>
The newer version of `SFT` class build the logic that the tokens are automatically ignored during loss computation when `completion_only_loss` or `assistant_only_loss` are set in the `SFTConfig` object.

In [38]:
from trl_data_collator import DataCollatorForCompletionOnlyLM

Let's see what is effect on the data collator.

In [39]:
response_template = '<|assistant|>'
complete_collator = DataCollatorForCompletionOnlyLM(response_template=response_template, tokenizer=tokenizer)
complete_loader = DataLoader(tokenized_dataset, batch_size=2,collate_fn=complete_collator)
complete_batch = next(iter(complete_loader))

We provided `'<|assistant|>'` to `response_template` field of `DataCollatorForCompletionOnlyLM`. It indicates where the <u>user's prompt</u> starts.

In [40]:
complete_batch

{'input_ids': tensor([[32010,   450, 29773,   305,   508,  7297,  2243,   333,   373,   278,
         10597,   715,  1331, 29889, 32007, 32001,  1551,   278, 10597,   715,
          1331, 29892,   278, 29773,   305,   508,  7297,  2243,   333, 29889,
          3869, 29892,   298, 21478,  1758, 29889, 32007, 32000],
        [32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000, 32000,
         32010,  8467,   434,   278,  9869,   304,   278,  6501,  7254,  3239,
         29889, 32007, 32001,  8467,   434,   278,  9869,   304,   278,  6501,
          7254,  3239, 29892,   366,  1818, 29889, 32007, 32000]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'labels': tensor([[ -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
   

****
There are more `-100` tokens in the `labels` field. And the second sentence of `input_ids` field was not padded.

In [42]:
valid_token_indexes = (complete_batch['labels'][0] >= 0)
valid_tokens = complete_batch['labels'][0][valid_token_indexes]
tokenizer.decode(valid_tokens)

'On the smooth planks, the birch canoe slid. Yes, hrrrm.<|end|>'